# Global 21cm Signal with Dark Photon Effects

Compute and visualize how dark photon conversions modify the sky-averaged 21cm signal. Shows the difference between standard $\rm{\Lambda CDM}$ and $\rm{\Lambda CDM}$ + dark photon scenarios.

## Setup

In [ ]:
import sys
sys.path.append("../")
sys.path.append("../21cmfast_sim/")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colormaps as cms
import matplotlib.pylab as pylab
from scipy.optimize import fsolve
from tqdm import tqdm

# Jupyter setup
%load_ext autoreload
%autoreload 2
%matplotlib inline

# Custom modules
from plot_params import params
from galaxy_survey import *

pylab.rcParams.update(params)
cols_default = plt.rcParams['axes.prop_cycle'].by_key()['color']

## Survey Setup

In [ ]:
# Same lightcone data as previous notebooks
cache_name = f'/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/v4_lightcones/'
lc_name = "LightCone_z5.5_HIIDIM=200_BOXLEN=300.0_r3843290498042390.h5"
lightconer_name = "lightconer_seed3843290498042390.pkl"

roman = Survey("Roman", "Lya", 
        cache=cache_name, 
        lc_name=lc_name, 
        lightconer_name=lightconer_name, 
        nside=2048, 
        foregrounds_path="/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_foregrounds_0.05mJy/all_foreground_maps.npy",
        use_pixel_ilc=True,
        )

## Global Signal Computation

In [ ]:
# Compute brightness temperature with dark photon conversions
roman.get_bt_plus_dp(mA=3e-14, epsilon=1e-6)

## Global Signal Comparison Plot

In [ ]:
# Plot sky-averaged (global) 21cm signal vs redshift
plt.plot(roman.rs_array[1:], np.average(roman.bt, axis=(0,1))[1:], label=r"$\Lambda$CDM")
plt.plot(roman.rs_array[1:], np.average(roman.bt_w_dp, axis=(0,1))[1:], label=r"$\Lambda$CDM + $\gamma \to A'$")

plt.xlabel(r"$z$")
plt.ylabel(r"$\langle T_{21}\rangle$ [mK]")
plt.legend()
plt.savefig("plots/global_signal.pdf", bbox_inches="tight")

In [ ]:
(2e-4)**(5/6)

## Homogenous Analytic Estimate

In [ ]:
Tgamma0 = 2.73e3
def Ptot(mA, eps):
    zobs = 17
    xe = 2e-4
    return 0.0006317 * (mA**2/(1e-14)**2)**(1/6) * (1 * (1. + zobs) * (eps/1e-7)**2)
mAs = np.geomspace(1e-15, 1e-12, 100)
eps_sols = np.array([fsolve(lambda eps: Tgamma0 * Ptot(mA, eps) - 150, 1e-7)[0] for mA in tqdm(mAs)])

In [ ]:
fsolve(lambda eps: Tgamma0 * Ptot(1e-14, eps) - 150, 1e-7)

In [ ]:
Tgamma0 * Ptot(1e-14, 1e-5)

In [ ]:
plt.semilogx(mAs, eps_sols)
plt.xscale("log")
plt.yscale("log")
plt.xscale